# Prédiction des Matchs de Tennis ATP (Kaggle Dataset)
Objectif du notebook : prédire le gagnant d’un match de tennis avec un modèle MLP binaire en exploitant l'affinité des joueurs selon les surfaces.

## 1. Imports
Importation des librairies nécessaires.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


## 2. Chargement des données
On lit le CSV avec `.read_csv('atp_tennis.csv')`.

In [ ]:
df = pd.read_csv("atp_tennis.csv", low_memory=False)

print("--- EXEMPLES DE NOMS DE JOUEURS DANS LA BASE ---")
# Afficher comment les noms sont formattés pour ne pas se tromper
print(df['Player_1'].value_counts().head(5))


## 3. Sélection et Nettoyage

In [ ]:
cols_to_keep = ['Surface', 'Player_1', 'Player_2', 'Winner', 'Rank_1', 'Rank_2']

df = df[cols_to_keep]
df = df.dropna()
print(f"Lignes après nettoyage : {len(df)}")


## 4. Création de la Target

In [ ]:
df['target'] = (df['Winner'] == df['Player_1']).astype(int)
print("Equilibre des cibles :\n", df['target'].value_counts(normalize=True))


## 5. Feature engineering : Aisance Surface LISSÉE (Bayesian Smoothing)
*Problème fréquent : Un joueur obscur qui a joué 1 seul match sur gazon et l'a gagné a 100% de win_rate, ce qui trompe le réseau (il le croirait meilleur que Federer qui a 87% sur 200 matchs).*\n\n**Solution** : On applique un lissage bayésien sur le `win_rate`. Les joueurs ayant peu de matchs verront leur moyenne tirée vers 50%. Les joueurs avec 100+ matchs auront leur vraie moyenne de révélée !

In [ ]:
# 1. Identifier le perdant
df['Loser'] = np.where(df['Winner'] == df['Player_1'], df['Player_2'], df['Player_1'])

# 2. Construire table historique
winners = df[['Winner', 'Surface']].copy()
winners['Victoire'] = 1
winners.columns = ['Player', 'Surface', 'Victoire']

losers = df[['Loser', 'Surface']].copy()
losers['Victoire'] = 0
losers.columns = ['Player', 'Surface', 'Victoire']

historique = pd.concat([winners, losers])

# Calcul des Sommes de Victoires et Nombres de Matchs
stats_surface = historique.groupby(['Player', 'Surface'])['Victoire'].agg(['sum', 'count']).reset_index()

# ---- BAYESIAN SMOOTHING ----
C = 10 # Coefficient de confiance (on "rajoute" virtuellement 10 matchs moyens à chaque joueur)
M = 0.5 # Moyenne globale d'un joueur lambda
stats_surface['win_rate_on_surface'] = (stats_surface['sum'] + C * M) / (stats_surface['count'] + C)
# ----------------------------

# 3. Injecter l'info
df = df.merge(stats_surface[['Player', 'Surface', 'win_rate_on_surface']], left_on=['Player_1', 'Surface'], right_on=['Player', 'Surface'], how='left')
df = df.rename(columns={'win_rate_on_surface': 'p1_surface_win_rate'})
df = df.drop(columns=['Player']) 

df = df.merge(stats_surface[['Player', 'Surface', 'win_rate_on_surface']], left_on=['Player_2', 'Surface'], right_on=['Player', 'Surface'], how='left')
df = df.rename(columns={'win_rate_on_surface': 'p2_surface_win_rate'})
df = df.drop(columns=['Player'])

df['p1_surface_win_rate'] = df['p1_surface_win_rate'].fillna(0.5)
df['p2_surface_win_rate'] = df['p2_surface_win_rate'].fillna(0.5)

# 4. Différence d'Aisance et de Classement
df['surface_affinity_diff'] = df['p1_surface_win_rate'] - df['p2_surface_win_rate']
df['rank_diff'] = df['Rank_2'] - df['Rank_1']

# One-hot encoding de la surface 
df_final = pd.get_dummies(df, columns=['Surface'])
df_final = df_final.drop(columns=['Player_1', 'Player_2', 'Winner', 'Loser'])

print(f"Dimensions de nos données : {df_final.shape}")
display(df_final.head())


## 6. Préparation des données

In [ ]:
y = df_final['target']
X = df_final.drop('target', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 7. Modèle MLP

In [ ]:
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


## 8. Entraînement

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
print("--- ENTRAÎNEMENT TERMINÉ ! ---")


## 9. Évaluation (Scores)

In [ ]:
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Accuracy : {accuracy:.4f}")


## 10. Visualisation

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']

epochs_range = range(1, len(acc) + 1)
plt.figure(figsize=(5, 4))
plt.plot(epochs_range, acc, label='Train')
plt.plot(epochs_range, val_acc, label='Validation')
plt.title('Accuracy')
plt.legend()
plt.show()


## 11. Fonction de recherche de nom exact
Pour utiliser le simulateur, vous devez entrer l'orthographe EXACTE du joueur. Voici un petit outil pour vous aider.

In [ ]:
def chercher_un_joueur(nom):
    # Cherche tous les joueurs dont le nom contient ce que vous tapez !
    res = historique[historique['Player'].str.contains(nom, case=False, na=False)]['Player'].unique()
    print(f"Résultats correspondants pour '{nom}' : {res}")

chercher_un_joueur("Djokovic")
chercher_un_joueur("Nadal")


## 12. SIMULATEUR DE MATCH (Prédiction Interactive)

In [ ]:
def simulateur_de_match(nom_joueur_1, nom_joueur_2, surface_choisie):
    
    # 1. Rang
    def get_latest_rank(player_name):
        r1 = df[df['Player_1'] == player_name]['Rank_1']
        r2 = df[df['Player_2'] == player_name]['Rank_2']
        if not r1.empty and not r2.empty:
            return r1.iloc[-1] if r1.index[-1] > r2.index[-1] else r2.iloc[-1]
        elif not r1.empty: return r1.iloc[-1]
        elif not r2.empty: return r2.iloc[-1]
        else: return np.nan
        
    rank_p1 = get_latest_rank(nom_joueur_1)
    rank_p2 = get_latest_rank(nom_joueur_2)
    
    if pd.isna(rank_p1): return f"Erreur : '{nom_joueur_1}' introuvable ! Lancez 'chercher_un_joueur' pour trouver la bonne orthographe."
    if pd.isna(rank_p2): return f"Erreur : '{nom_joueur_2}' introuvable ! Lancez 'chercher_un_joueur' pour trouver la bonne orthographe."
    
    # 2. Aisance lissée
    def get_surface_win_rate(player_name, surf):
        row = stats_surface[(stats_surface['Player'] == player_name) & (stats_surface['Surface'] == surf)]
        if not row.empty: return row['win_rate_on_surface'].values[0]
        return 0.5 
        
    win_rate_p1 = get_surface_win_rate(nom_joueur_1, surface_choisie)
    win_rate_p2 = get_surface_win_rate(nom_joueur_2, surface_choisie)
    
    # 3. Features calculées
    diff_affinite = win_rate_p1 - win_rate_p2
    diff_classement = rank_p2 - rank_p1
    
    # 4. DataFrame Inférence
    cols = X.columns
    match_data = pd.DataFrame(np.zeros((1, len(cols))), columns=cols)
    match_data['surface_affinity_diff'] = diff_affinite
    match_data['rank_diff'] = diff_classement
    
    col_surface_name = f'Surface_{surface_choisie}'
    if col_surface_name in match_data.columns:
        match_data[col_surface_name] = 1.0
        
    match_scaled = scaler.transform(match_data)
    
    # 5. Prédiction
    prob_p1_wins = model.predict(match_scaled, verbose=0)[0][0]
    
    print(f"============================================")
    print(f" 🎾 {nom_joueur_1} VS {nom_joueur_2} sur {surface_choisie.upper()}")
    print(f"============================================")
    print(f"- {nom_joueur_1} | Aisance Surface (Lissée): {win_rate_p1*100:.1f}% | Classement: {rank_p1:.0f}")
    print(f"- {nom_joueur_2} | Aisance Surface (Lissée): {win_rate_p2*100:.1f}% | Classement: {rank_p2:.0f}")
    print(f"\n> Probabilité analytique que {nom_joueur_1} gagne : {prob_p1_wins*100:.1f}%")

# Test par défaut :
simulateur_de_match('Federer R.', 'Nadal R.', 'Clay')


## 13. Sauvegarde

In [ ]:
model.save("tennis_model.h5")
print("Modèle exporté !")
